# Phase 1: Data Ingestion & Preprocessing

This notebook handles the initial setup of the data pipeline.

**Goals:**
1. Setup Kaggle API and download the dataset.
2. Preprocess images (Resize to 128x128, Normalize).
3. Split data into `train_normal`, `train_defect`, `valid_mixed`, `test_mixed`.

**Prerequisites:**
- You need a Kaggle account.
- Create a new API token: Account -> Create New API Token -> downloads `kaggle.json`.
- Upload `kaggle.json` when prompted.

In [ ]:
# 1. Install Kaggle Library
!pip install -q kaggle

In [ ]:
# 2. Upload kaggle.json
from google.colab import files
import os

if not os.path.exists('kaggle.json'):
    print("Please upload your kaggle.json file")
    uploaded = files.upload()
    
    for fn in uploaded.keys():
        print('User uploaded file "{name}" with length {length} bytes'.format(
            name=fn, length=len(uploaded[fn])))
        # Move to /root/.kaggle/kaggle.json
        !mkdir -p ~/.kaggle/
        !cp kaggle.json ~/.kaggle/
        !chmod 600 ~/.kaggle/kaggle.json
else:
    print("kaggle.json already exists")

In [ ]:
# 3. Download Dataset
# TODO: REPLACE THIS WITH YOUR DATASET NAME
DATASET_NAME = "nexuswho/fabric-defects-dataset" # User specified dataset

print(f"Downloading {DATASET_NAME}...")
!kaggle datasets download -d {DATASET_NAME}

# Unzip
zip_name = DATASET_NAME.split('/')[-1] + ".zip"
!unzip -q {zip_name} -d raw_data
print("Dataset unzipped to raw_data/")

In [ ]:
import cv2
import numpy as np
import os
import shutil
from sklearn.model_selection import train_test_split
import glob
import matplotlib.pyplot as plt

# Configuration
IMG_SIZE = 128  # 128x128 for DCGAN stability
RANDOM_SEED = 42

# Define paths (adjust based on dataset structure)
# Assuming standard structure: raw_data/Defect_images/ and raw_data/Normal_images/
# You might need to adjust these paths after inspecting 'raw_data/'
BASE_DIR = 'raw_data'

# Helper to find image files
def list_images(directory):
    extensions = ['*.jpg', '*.jpeg', '*.png', '*.bmp']
    files = []
    for ext in extensions:
        files.extend(glob.glob(os.path.join(directory, '**', ext), recursive=True))
    return files

print("Inspecting raw_data structure...")
for root, dirs, files in os.walk(BASE_DIR):
    level = root.replace(BASE_DIR, '').count(os.sep)
    indent = ' ' * 4 * (level)
    print(f"{indent}{os.path.basename(root)}/")
    subindent = ' ' * 4 * (level + 1)
    if len(files) > 0:
        print(f"{subindent}{len(files)} files")


In [ ]:
# 4. Data Loading Logic
# CHANGE THESE PATHS based on the structure printed above
NORMAL_DIR = os.path.join(BASE_DIR, 'match_pattern_xyz_normal') # Example path
DEFECT_DIR = os.path.join(BASE_DIR, 'match_pattern_xyz_defect') # Example path
# If dataset is 'textiledefectdetection':
# It usually has 'match_pattern...' folders. We need to aggregate them.

# Aggregating all images
all_normal_imgs = []
all_defect_imgs = []

for root, dirs, files in os.walk(BASE_DIR):
    # Logic for nexuswho dataset (horizontal, vertical, holes -> defect)
    for file in files:
        if file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
            full_path = os.path.join(root, file)
            lower_path = full_path.lower()
            
            if 'mask' in lower_path: continue # Exclude masks
            
            if any(k in lower_path for k in ['content/horizontal', 'content/vertical', 'content/holes', 'captured/horizontal', 'captured/vertical', 'captured/holes', 'horizontal', 'vertical', 'holes', 'defect']):
                all_defect_imgs.append(full_path)
            elif any(k in lower_path for k in ['normal', 'ok', 'noobject', 'good']):
                all_normal_imgs.append(full_path)
            # Else ignored

print(f"Found {len(all_normal_imgs)} Normal images")
print(f"Found {len(all_defect_imgs)} Defect images")

In [ ]:
# 5. Preprocessing & Splitting
# Create directories
PROCESSED_DIR = 'processed_data'

DIRS = {
    'train_normal': os.path.join(PROCESSED_DIR, 'train_normal'),
    'train_defect': os.path.join(PROCESSED_DIR, 'train_defect'),
    'valid_mixed': os.path.join(PROCESSED_DIR, 'valid_mixed'),
    'test_mixed': os.path.join(PROCESSED_DIR, 'test_mixed')
}

for d in DIRS.values():
    os.makedirs(d, exist_ok=True)
    # Create subdirs for labeled sets (valid/test)
    if 'mixed' in d:
        os.makedirs(os.path.join(d, 'normal'), exist_ok=True)
        os.makedirs(os.path.join(d, 'defect'), exist_ok=True)

def preprocess_and_save(img_path, save_path):
    img = cv2.imread(img_path)
    if img is None:
        return False
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    # We save as PNG to avoid compression artifacts, normalization happens at load time (dataloader)
    cv2.imwrite(save_path, img)
    return True

# Splitting Strategy
# 1. Isolate test set (10%)
# 2. Isolate validation set (10%)
# 3. Remainder is training

def split_and_process(file_list, category):
    train_files, test_files = train_test_split(file_list, test_size=0.2, random_state=RANDOM_SEED)
    test_files, valid_files = train_test_split(test_files, test_size=0.5, random_state=RANDOM_SEED) # 10% valid, 10% test

    print(f"Processing {category}: {len(train_files)} Train, {len(valid_files)} Valid, {len(test_files)} Test")

    # Save Train
    if category == 'normal':
        dest_dir = DIRS['train_normal']
    else:
        dest_dir = DIRS['train_defect']

    count = 0
    for f in train_files:
        if preprocess_and_save(f, os.path.join(dest_dir, f"{category}_{count}.png")):
            count += 1

    # Save Valid & Test (Mixed folders)
    for f in valid_files:
        preprocess_and_save(f, os.path.join(DIRS['valid_mixed'], category, f"{category}_{os.path.basename(f)}"))
        
    for f in test_files:
        preprocess_and_save(f, os.path.join(DIRS['test_mixed'], category, f"{category}_{os.path.basename(f)}"))

split_and_process(all_normal_imgs, 'normal')
split_and_process(all_defect_imgs, 'defect')

print("Data preprocessing complete!")

In [ ]:
# 6. Verify processed data
import random
from PIL import Image

train_defect_path = DIRS['train_defect']
sample_files = os.listdir(train_defect_path)[:5]

plt.figure(figsize=(15, 5))
for i, file in enumerate(sample_files):
    img = Image.open(os.path.join(train_defect_path, file))
    plt.subplot(1, 5, i+1)
    plt.imshow(img)
    plt.title(f"Defect {i}")
    plt.axis('off')
plt.show()